In [1]:
import os
from dataclasses import dataclass, field
from typing import List, Dict, Callable, Optional, Tuple, Any
from datetime import timedelta

import pandas as pd

# --- Imports for PDF ---
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm

# ============================================================
# PDF helpers (TEXT ONLY) - PARAGRAPH OUTPUT
# ============================================================

def report_text_to_pdf(
    report_text: str,
    output_path: str = "rapor.pdf",
    font_path: str | None = None,
    title: str | None = None
) -> str:
    base_font = "Helvetica"
    if font_path:
        if not os.path.exists(font_path):
            raise FileNotFoundError(f"Font file not found: {font_path}")

        from reportlab.pdfbase import pdfmetrics
        from reportlab.pdfbase.ttfonts import TTFont



        pdfmetrics.registerFont(TTFont("CustomFont", font_path))
        base_font = "CustomFont"

    doc = SimpleDocTemplate(
        output_path,
        pagesize=A4,
        leftMargin=2 * cm, rightMargin=2 * cm, topMargin=2 * cm, bottomMargin=2 * cm
    )

    styles = getSampleStyleSheet()

    body = ParagraphStyle(
        "Body",
        parent=styles["Normal"],
        fontName=base_font,
        fontSize=10.5,
        leading=14,
        alignment=4  # justify
    )

    title_style = ParagraphStyle(
        "Title",
        parent=styles["Heading1"],
        fontName=base_font,
        alignment=1  # center
    )

    story = []

    if title:
        safe_title = title.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
        story.append(Paragraph(safe_title, title_style))
        story.append(Spacer(1, 12))

   # Split into paragraphs: empty line as paragraph separator
    lines = [ln.rstrip() for ln in report_text.splitlines()]
    paragraphs: List[str] = []
    buf: List[str] = []

    for ln in lines:
        if ln.strip() == "":
            if buf:
                paragraphs.append(" ".join(buf))
                buf = []
        else:
            buf.append(ln.strip())

    if buf:
        paragraphs.append(" ".join(buf))

    for p in paragraphs:
        safe_p = p.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
        story.append(Paragraph(safe_p, body))
        story.append(Spacer(1, 8))

    doc.build(story)
    return output_path


# ============================================================
# 0) Parameters
# ============================================================

@dataclass
class OEEConfig:

    OEE_LOW_TRAP: Tuple[float, float, float, float] = (41.71, 41.71, 41.71, 68.79)
    OEE_MID_TRI:  Tuple[float, float, float]        = (41.71, 68.79, 89.85)
    OEE_HIGH_TRAP: Tuple[float, float, float, float] = (68.79, 89.85, 89.85, 89.85)

    OEE_D_LOW_TRAP:  Tuple[float, float, float, float] = (22.37, 22.37, 22.37, 61.68) 
    OEE_D_MID_TRI:   Tuple[float, float, float]        = (22.37, 61.68, 90.10)
    OEE_D_HIGH_TRAP: Tuple[float, float, float, float] = (61.68, 90.10, 90.10, 90.10)

    AVAIL_LOW_TRAP: Tuple[float, float, float, float] = (89.08, 89.08, 89.08, 96.15)
    AVAIL_MID_TRI:  Tuple[float, float, float]        = (89.08, 96.15, 99.80)
    AVAIL_HIGH_TRAP: Tuple[float, float, float, float] = (96.15, 99.80, 99.80, 99.80)

    PERF_LOW_TRAP: Tuple[float, float, float, float] = (42.40, 42.40, 42.40, 69.75)
    PERF_MID_TRI:  Tuple[float, float, float]        = (42.40, 69.75, 90.04)
    PERF_HIGH_TRAP: Tuple[float, float, float, float] = (69.75, 90.04, 90.04, 90.04)

    QUAL_LOW_TRAP: Tuple[float, float, float, float] = (89.00, 89.00, 89.00, 96.88)
    QUAL_MID_TRI:  Tuple[float, float, float]        = (89.00, 96.88, 99.99)
    QUAL_HIGH_TRAP: Tuple[float, float, float, float] = (96.88, 99.99, 99.99, 99.99)

    UTIL_LOW_TRAP: Tuple[float, float, float, float] = (35.53, 35.53, 35.53, 50.44)
    UTIL_MID_TRI:  Tuple[float, float, float]        = (35.53, 50.44, 68.14)
    UTIL_HIGH_TRAP: Tuple[float, float, float, float] = (50.44, 68.14, 68.14, 68.14)

    OPE_LOWER_TRAP:  Tuple[float, float, float, float] = (-26.99, -26.99, -26.99, -1.58)
    OPE_SAME_TRI:   Tuple[float, float, float]        = (-26.99, -1.58, 17.50)
    OPE_HIGHER_TRAP: Tuple[float, float, float, float] = (-1.58, 17.50, 17.50, 17.50)

    PLANT_LOWER_TRAP:  Tuple[float, float, float, float] = (-24.24, -24.24, -24.24, 1.67)
    PLANT_SAME_TRI:   Tuple[float, float, float]        = (-24.24, 1.67, 22.76)
    PLANT_HIGHER_TRAP: Tuple[float, float, float, float] = (1.67, 22.76, 22.76, 22.76)

    CLW_DETERIOR_TRAP:  Tuple[float, float, float, float] = (-17.71, -17.71, -17.71, 0.44)
    CLW_UNCHANGE_TRI:   Tuple[float, float, float]        = (-17.71, 0.44, 22.28)
    CLW_IMPROVE_TRAP: Tuple[float, float, float, float] = (0.44, 22.28, 22.28, 22.28)
    
    TREND_DEC_TRAP: Tuple[float, float, float, float] = (-8.69, -8.69, -8.69, -0.055)
    TREND_STB_TRI: Tuple[float, float, float] = (-8.69, -0.055, 8.72)
    TREND_INC_TRAP: Tuple[float, float, float, float] = (-0.055, 8.72, 8.72, 8.72)

    VAR_LOW_TRAP: Tuple[float, float, float, float] = (1.01, 1.01, 1.01, 2.02)
    VAR_MID_TRI: Tuple[float, float, float]         = (1.01, 2.02, 3.17)
    VAR_HIGH_TRAP: Tuple[float, float, float, float] = (2.02, 3.17, 3.17, 3.17)

    SD_FEW_TRAP: Tuple[float, float, float, float]  = (0.00, 0.00, 0.30, 0.50)
    SD_MOST_TRI: Tuple[float, float, float]         = (0.30, 0.60, 0.85)
    SD_ALL_TRAP: Tuple[float, float, float, float]  = (0.80, 0.90, 1.00, 1.00)

    AP_OUT_LOW_TRAP: Tuple[float, float, float, float] = (0.0, 0.0, 2.0, 4.0)
    AP_OUT_MID_TRI: Tuple[float, float, float] = (3.0, 5.0, 7.0)
    AP_OUT_HIGH_TRAP: Tuple[float, float, float, float] = (6.0, 8.0, 10.0, 10.0)

    # (trend, var) -> process (Mamdani)
    MAMDANI_RULES: List[Dict[str, str]] = field(default_factory=lambda: [
        {"trend": "increasing",  "var": "low",  "out": "stable"},
        {"trend": "increasing",  "var": "medium",  "out": "partly stable"},
        {"trend": "increasing",  "var": "high", "out": "unstable"},

        {"trend": "constant",  "var": "low",  "out": "stable"},
        {"trend": "constant",  "var": "medium",  "out": "partly stable"},
        {"trend": "constant",  "var": "high", "out": "unstable"},

        {"trend": "decreasing", "var": "low",  "out": "partly stable"},
        {"trend": "decreasing", "var": "medium",  "out": "unstable"},
        {"trend": "decreasing", "var": "high", "out": "unstable"},
    ])

    # ========================================================
    # Relevance (r) vectors
    # ========================================================

    R_OEE: Tuple[float, float, float] = (1.0, 1.0, 1.0)
    R_AVAIL: Tuple[float, float, float] = (1.0, 1.0, 0)
    R_PERF: Tuple[float, float, float]  = (1.0, 1.0, 0)
    R_QUAL: Tuple[float, float, float]  = (1.0, 1.0, 0)
    R_UTIL: Tuple[float, float, float] = (1.0, 1.0, 0)
    R_C_PLT: Tuple[float, float, float] = (1.0, 1.0, 1.0)
    R_C_OPE: Tuple[float, float, float] = (1.0, 1.0, 1.0)
    R_SD: Tuple[float, float, float] = (0.2, 0.8, 1.0) 
    R_DAY: Tuple[float, float, float] = (1.0, 0.7, 0.5) 
    R_C_LW: Tuple[float, float, float] = (1.0, 1.0, 1.0)
    R_TREND: Tuple[float, float, float] = (1.0, 1.0, 1.0)
    R_VAR: Tuple[float, float, float] = (1.0, 1.0, 1.0)
    R_PROCESS: Tuple[float, float, float] = (1.0, 1.0, 1.0)


# ============================================================
# 1) CP 
# ============================================================

@dataclass

class CP:
    name: str
    labels: List[str]
    w: List[float] = field(default_factory=list)  # validity degrees
    r: List[float] = field(default_factory=list)  # relevance degrees
    label_to_idx: Dict[str, int] = field(init=False, default_factory=dict)

    def __post_init__(self):
        if not self.w:
            self.w = [0.0 for _ in self.labels]
        if not self.r:
            self.r = [1.0 for _ in self.labels]

        self.label_to_idx = {lbl: i for i, lbl in enumerate(self.labels)}
        if len(self.label_to_idx) != len(self.labels):
            raise ValueError(f"{self.name}: duplicate entries in labels, index map conflicts.")

    def reset(self, reset_relevance: bool = False):
        self.w = [0.0 for _ in self.labels]
        if reset_relevance:
            self.r = [1.0 for _ in self.labels]

    def set_degrees(self, degrees):

        if degrees is None:
            self.w = []
            return
        if len(degrees) == 0:
            self.w = []
            return

        first = degrees[0]

    
        if isinstance(first, (list, tuple)):
            for row in degrees:
                if len(row) != len(self.labels):
                    raise ValueError("Number of labels does not match the matrix row.")
            self.w = [list(map(float, row)) for row in degrees]
            return

        if len(degrees) != len(self.labels):
            raise ValueError("Number of labels does not match.")
        self.w = list(map(float, degrees))

    def set_relevance(self, relevance: List[float]):
        if len(relevance) != len(self.labels):
            raise ValueError(f"Relevance size does not match. labels={len(self.labels)} r={len(relevance)}")
        self.r = relevance

    def degree_of(self, label: str) -> float:
        return self.w[self.label_to_idx[label]]

    def relevance_of(self, label: str) -> float:
        return self.r[self.label_to_idx[label]]

    def best_label_wr(self, rel_threshold: float = 0.0) -> str:
        scores: List[float] = []
        for wi, ri in zip(self.w, self.r):
            scores.append(0.0 if ri < rel_threshold else wi * ri)

        if all(s == 0.0 for s in scores):
            idx = max(range(len(self.w)), key=lambda i: self.w[i])
        else:
            idx = max(range(len(scores)), key=lambda i: scores[i])

        return self.labels[idx]


# ============================================================
# 2) Membership functions
# ============================================================

class TriangleMF:
    def __init__(self, a, b, c):
        self.a, self.b, self.c = a, b, c

    def __call__(self, x: float) -> float:
        a, b, c = self.a, self.b, self.c
        if x <= a or x >= c:
            return 0.0
        if a < x < b:
            return (x - a) / (b - a) if b != a else 0.0
        if b < x < c:
            return (c - x) / (c - b) if c != b else 0.0
        return 1.0


class TrapezoidMF:
    def __init__(self, a, b, c, d):
        self.a, self.b, self.c, self.d = a, b, c, d

    def __call__(self, x: float) -> float:
        a, b, c, d = self.a, self.b, self.c, self.d

        # left shoulder
        if a == b and c < d:
            if x <= a:
                return 1.0
            if a < x <= c:
                return 1.0
            if c < x < d:
                return (d - x) / (d - c) if d != c else 0.0
            return 0.0

        # right shoulder
        if c == d and a < b:
            if x <= a:
                return 0.0
            if a < x < b:
                return (x - a) / (b - a) if b != a else 0.0
            if b <= x:
                return 1.0
            return 0.0

        # general trapezoid
        if x <= a or x >= d:
            return 0.0
        if a < x < b:
            return (x - a) / (b - a) if b != a else 0.0
        if b <= x <= c:
            return 1.0
        if c < x < d:
            return (d - x) / (d - c) if d != c else 0.0
        return 0.0


@dataclass
class FuzzyPartition:
    label_to_mf: Dict[str, Callable[[float], float]]

    def degrees(self, x: float, order: List[str]) -> List[float]:
        return [self.label_to_mf[label](x) for label in order]


# ============================================================
# 4) LDCP
# ============================================================

@dataclass
class CandidateSentence:
    protoform_id: str
    label: str
    text: str
    validity: float               
    relevance: float = 1.0        
    selection_value: float = 0.0  


def candidates_to_df(cands: List[CandidateSentence]) -> pd.DataFrame:
    rows = []
    for i, c in enumerate(cands, 1):
        rows.append({
            "Order No": i,
            "Protoform": c.protoform_id,
            "Label": c.label,
            "Candidate Sentence": c.text,
            "Validity (W)": round(float(c.validity), 4),
            "Relevance (R)": round(float(c.relevance), 4),
            "Selection Value": round(float(c.selection_value), 6)
        })
    return pd.DataFrame(rows, columns=[
        "Order No", "Protoform", "Label", "Candidate Sentence",
        "Validity (W)", "Relevance (R)", "Selection Value"
    ])


def audit_to_df(audit_rows: List[Tuple[str, float, float, float, bool, str]]) -> pd.DataFrame:
    rows = []
    for i, (pf_id, W, R, SV, inc, reason) in enumerate(audit_rows, 1):
        rows.append({
            "Order No": i,
            "Protoform": pf_id,
            "Validity (W)": round(float(W), 4),
            "Relevance (R)": round(float(R), 4),
            "Selection Value": round(float(SV), 6),
            "Included": bool(inc),
            "Reason": str(reason),
        })
    return pd.DataFrame(rows, columns=[
        "Order No", "Protoform", "Validity (W)", "Relevance (R)", "Selection Value", "Included", "Reason"
    ])


# ============================================================
# 5) PM layer + Report template
# ============================================================

@dataclass
class PMResult:
    produced_cps: List[CP] = field(default_factory=list)
    produced_candidates: List[CandidateSentence] = field(default_factory=list)
    produced_scalars: Dict[str, Any] = field(default_factory=dict)

# ============================================================
# Protoform Inclusion Rule
# ============================================================
@dataclass
class ProtoformRule:
    mode: str = "conditional"   # "mandatory" or "conditional"
    W_min: float = 0.0          # validity threshold
    R_min: float = 0.0          # relevance threshold
    SV_min: float = 0.0         # selection value threshold
    include_if: Optional[Callable[[CandidateSentence, Dict[str, Any]], bool]] = None
class PM:
    pm_id: str

    def __init__(self, pm_id: str):
        self.pm_id = pm_id

    def compute(self, data: Dict[str, Any], cp_store: Dict[str, CP]) -> PMResult:
        raise NotImplementedError


class ReportTemplate:
 
    def __init__(self, cfg: OEEConfig):
        self.cfg = cfg

    def _norm_sentence(self, s: str) -> str:
        s = (s or "").strip()
        if not s:
            return ""
        if s[-1] not in ".!?":
            s += "."
        return s

    def format(self, selected: Dict[str, CandidateSentence], machine_id: str) -> str:
        parts: List[str] = []

        def add_if(key: str):
            if key in selected:
                parts.append(self._norm_sentence(selected[key].text))
        

        # 1) OEE Level
        add_if("pf_CP_OEE")
        
        # 2) Avail/Perf/Qual
        used = set()

        # If there are three:
        if ("pf_CP_Avail" in selected) and ("pf_CP_Perf" in selected) and ("pf_CP_Qual" in selected):
            a = selected["pf_CP_Avail"]
            p = selected["pf_CP_Perf"]
            q = selected["pf_CP_Qual"]

            apq_combined = (
                f"When the OEE components are examined, the machine’s weekly availability is {a.label}, "
                f"its performance is {p.label}, and "
                f"its quality is at the {q.label} level."

            )
            parts.append(self._norm_sentence(apq_combined))
            used.update(["pf_CP_Avail", "pf_CP_Perf", "pf_CP_Qual"])

        # Pairs
        elif ("pf_CP_Avail" in selected) and ("pf_CP_Perf" in selected):
            a = selected["pf_CP_Avail"]
            p = selected["pf_CP_Perf"]

            ap_combined = (
                f"When the OEE components are examined, the machine’s weekly availability is {a.label} and "
                f"its performance is {p.label}."
            )
            parts.append(self._norm_sentence(ap_combined))
            used.update(["pf_CP_Avail", "pf_CP_Perf"])

        elif ("pf_CP_Perf" in selected) and ("pf_CP_Qual" in selected):
            p = selected["pf_CP_Perf"]
            q = selected["pf_CP_Qual"]

            pq_combined = (
                f"When the OEE components are examined, the machine’s weekly performance is {p.label} and "
                f"its quality is {q.label}."
            )
            parts.append(self._norm_sentence(pq_combined))
            used.update(["pf_CP_Perf", "pf_CP_Qual"])

        elif ("pf_CP_Avail" in selected) and ("pf_CP_Qual" in selected):
            a = selected["pf_CP_Avail"]
            q = selected["pf_CP_Qual"]

            aq_combined = (
                f"When the OEE components are examined, the machine’s weekly availability is {a.label}  and "
                f"its quality is {q.label}."

            )
            parts.append(self._norm_sentence(aq_combined))
            used.update(["pf_CP_Avail", "pf_CP_Qual"])

        #  Add one by one
        if ("pf_CP_Avail" in selected) and ("pf_CP_Avail" not in used):
            add_if("pf_CP_Avail")

        if ("pf_CP_Perf" in selected) and ("pf_CP_Perf" not in used):
            add_if("pf_CP_Perf")

        if ("pf_CP_Qual" in selected) and ("pf_CP_Qual" not in used):
            add_if("pf_CP_Qual")
  

        # 3) Weekly summary of days (SD)
        add_if("pf_CP_SD")

        # 4) Trend + variability + process combined sentence
        if ("pf_CP_Trend" in selected) and ("pf_CP_Var" in selected) and ("pf_CP_Process" in selected):
            tr = selected["pf_CP_Trend"]
            vr = selected["pf_CP_Var"]
            pr = selected["pf_CP_Process"]

            combined = (
                f"This week, since the OEE trend indicates a {tr.label} trend and "
                f"{vr.label} trend variability, "
                f"the machine’s production process is assessed as {pr.label}."
            )
            parts.append(self._norm_sentence(combined))
        else:
            add_if("pf_CP_Trend")
            add_if("pf_CP_Var")
            add_if("pf_CP_Process")
       
        
        # 5-6) Operation group + plant comparison (combine in a single sentence)
        if ("pf_CP_C_OPE" in selected) and ("pf_CP_C_PLT" in selected):
            co = selected["pf_CP_C_OPE"]
            cp = selected["pf_CP_C_PLT"]

            combined_compare = (
            f"The machine’s OEE level is {co.label} relative to its operation group, "
            f"and {cp.label} compared to the overall plant average."
          )
            parts.append(self._norm_sentence(combined_compare))
        else:
            add_if("pf_CP_C_OPE")
            add_if("pf_CP_C_PLT")

        # 7) Change
        add_if("pf_CP_C_LW")
        # 8) Util
        add_if("pf_CP_Util")
       # 2b) Action Priority
        add_if("pf_CP_ActionPriority")
        return " ".join([p for p in parts if p])


# ============================================================
# 6) PM’s
# ============================================================

def week_avg(days):
    
    if days is None or len(days) == 0:
        raise ValueError("cur_week_avg: 'days' must be a non-empty list/sequence")
    return sum(days) / len(days)

class PM_OEE(PM):
    def __init__(
        self,
        partition_OEE: FuzzyPartition,
        machine_id_getter: Callable[[Dict[str, Any]], str],
        out_cp_key: str = "CP_OEE"
    ):
        super().__init__("PM_OEE")
        self.partition = partition_OEE
        self.machine_id_getter = machine_id_getter
        self.out_cp_key = out_cp_key


    def compute(self, data: Dict[str, Any], cp_store: Dict[str, CP]) -> PMResult:
        oee_days = data["oee_days"]
        cur_week_avg = week_avg(oee_days)
        cp = cp_store[self.out_cp_key]
        cp.set_degrees(self.partition.degrees(cur_week_avg, cp.labels))

        machine_id = self.machine_id_getter(data)

        def tpl(lbl: str, validity: float) -> str:
            return f"The machine’s average OEE level for this week is {lbl}."
                   
        cands: List[CandidateSentence] = []
        for lbl, wi in zip(cp.labels, cp.w):
            validity = float(wi)
            ri = float(cp.relevance_of(lbl))
            selection_value = validity * ri
            cands.append(CandidateSentence("pf_CP_OEE", lbl, tpl(lbl, validity), validity, ri, selection_value))

        return PMResult(produced_cps=[cp], produced_candidates=cands, produced_scalars={})



class PM_Avail(PM):
    def __init__(
        self,
        partition_AVAIL: FuzzyPartition,
        out_cp_key: str = "CP_Avail"
    ):
        super().__init__("PM_Avail")
        self.partition = partition_AVAIL
        self.out_cp_key = out_cp_key

    def compute(self, data: Dict[str, Any], cp_store: Dict[str, CP]) -> PMResult:
        days = data["avail_days"]
        cur_week_avg = week_avg(days)
        cp = cp_store[self.out_cp_key]
        cp.set_degrees(self.partition.degrees(cur_week_avg, cp.labels))

        def tpl(lbl: str, validity: float) -> str:
            return f"When the OEE components are examined, the machine’s weekly availability is at the {lbl} level."
             

        cands: List[CandidateSentence] = []
        for lbl, wi in zip(cp.labels, cp.w):
            validity = float(wi)
            ri = float(cp.relevance_of(lbl))
            selection_value = validity * ri
            cands.append(CandidateSentence("pf_CP_Avail", lbl, tpl(lbl, validity), validity, ri, selection_value))

        return PMResult(
            produced_cps=[cp],
            produced_candidates=cands,
            produced_scalars={}
        )


class PM_Perf(PM):
    def __init__(
        self,
        partition_PERF: FuzzyPartition,
        out_cp_key: str = "CP_Perf"
    ):
        super().__init__("PM_Perf")
        self.partition = partition_PERF
        self.out_cp_key = out_cp_key

    def compute(self, data: Dict[str, Any], cp_store: Dict[str, CP]) -> PMResult:
        days = data["perf_days"]
        cur_week_avg = week_avg(days)
        cp = cp_store[self.out_cp_key]
        cp.set_degrees(self.partition.degrees(cur_week_avg, cp.labels))

        def tpl(lbl: str, validity: float) -> str:
            return f"When the OEE components are examined, the machine’s weekly performance is at the {lbl} level."
              

        cands: List[CandidateSentence] = []
        for lbl, wi in zip(cp.labels, cp.w):
            validity = float(wi)
            ri = float(cp.relevance_of(lbl))
            selection_value = validity * ri
            cands.append(CandidateSentence("pf_CP_Perf", lbl, tpl(lbl, validity), validity, ri, selection_value))

        return PMResult(
            produced_cps=[cp],
            produced_candidates=cands,
            produced_scalars={}
        )


class PM_Qual(PM):
    def __init__(
        self,
        partition_QUAL: FuzzyPartition,
        out_cp_key: str = "CP_Qual"
    ):
        super().__init__("PM_Qual")
        self.partition = partition_QUAL
        self.out_cp_key = out_cp_key

    def compute(self, data: Dict[str, Any], cp_store: Dict[str, CP]) -> PMResult:
        days = data["qual_days"]
        cur_week_avg = week_avg(days)
        cp = cp_store[self.out_cp_key]
        cp.set_degrees(self.partition.degrees(cur_week_avg, cp.labels))

        def tpl(lbl: str, validity: float) -> str:
            return f"When the OEE components are examined, the machine’s weekly quality is at the {lbl} level."
         

        cands: List[CandidateSentence] = []
        for lbl, wi in zip(cp.labels, cp.w):
            validity = float(wi)
            ri = float(cp.relevance_of(lbl))
            selection_value = validity * ri
            cands.append(CandidateSentence("pf_CP_Qual", lbl, tpl(lbl, validity), validity, ri, selection_value))

        return PMResult(
            produced_cps=[cp],
            produced_candidates=cands,
            produced_scalars={}
        )


class PM_Util(PM):
    def __init__(
        self,
        partition_UTIL: FuzzyPartition,
        out_cp_key: str = "CP_Util"
    ):
        super().__init__("PM_Util")
        self.partition = partition_UTIL
        self.out_cp_key = out_cp_key

    def compute(self, data: Dict[str, Any], cp_store: Dict[str, CP]) -> PMResult:
        days = data["util_days"]
        cur_week_avg = week_avg(days)
        cp = cp_store[self.out_cp_key]
        cp.set_degrees(self.partition.degrees(cur_week_avg, cp.labels))

        def tpl(lbl: str, validity: float) -> str:
            return f"The machine’s weekly capacity utilization rate is at the {lbl} level."
                   

        cands: List[CandidateSentence] = []
        for lbl, wi in zip(cp.labels, cp.w):
            validity = float(wi)
            ri = float(cp.relevance_of(lbl))
            selection_value = validity * ri
            cands.append(CandidateSentence("pf_CP_Util", lbl, tpl(lbl, validity), validity, ri, selection_value))

        return PMResult(
            produced_cps=[cp],
            produced_candidates=cands,
            produced_scalars={}
        )


class PM_C_PLT(PM):
    def __init__(
        self,
        partition_delta: FuzzyPartition,
        out_cp_key: str = "CP_C_PLT"
    ):
        super().__init__("PM_C_PLT")
        self.partition = partition_delta
        self.out_cp_key = out_cp_key

    def compute(self, data: Dict[str, Any], cp_store: Dict[str, CP]) -> PMResult:
        oee_days = data["oee_days"]
        plant_days = data["plant_days"]

        cur_week_avg = week_avg(oee_days)
        plant_ref_avg = week_avg(plant_days)
        delta_plant = cur_week_avg - plant_ref_avg
        data["delta_plant"] = delta_plant

        cp = cp_store[self.out_cp_key]
        cp.set_degrees(self.partition.degrees(delta_plant, cp.labels))

        def tpl(lbl: str, validity: float) -> str:
            return f"It is {lbl} relative to the overall plant average."
           

        cands: List[CandidateSentence] = []
        for lbl, wi in zip(cp.labels, cp.w):
            validity = float(wi)
            ri = float(cp.relevance_of(lbl))
            selection_value = validity * ri
            cands.append(CandidateSentence("pf_CP_C_PLT", lbl, tpl(lbl, validity), validity, ri, selection_value))

        return PMResult(produced_cps=[cp], produced_candidates=cands, produced_scalars={"delta_plant": delta_plant})


class PM_C_OPE(PM):
    def __init__(
        self,
        partition_delta: FuzzyPartition,
        out_cp_key: str = "CP_C_OPE"
    ):
        super().__init__("PM_C_OPE")
        self.partition = partition_delta
        self.out_cp_key = out_cp_key

    def compute(self, data: Dict[str, Any], cp_store: Dict[str, CP]) -> PMResult:
        oee_days = data["oee_days"]
        ope_days = data["ope_days"]

        cur_week_avg = week_avg(oee_days)
        ope_ref_avg = week_avg(ope_days)
        delta_ope = cur_week_avg - ope_ref_avg
        data["delta_ope"] = delta_ope

        cp = cp_store[self.out_cp_key]
        cp.set_degrees(self.partition.degrees(delta_ope, cp.labels))

        def tpl(lbl: str, validity: float) -> str:
            return f"The machine is operating at a {lbl} level relative to its operation group."
          

        cands: List[CandidateSentence] = []
        for lbl, wi in zip(cp.labels, cp.w):
            validity = float(wi)
            ri = float(cp.relevance_of(lbl))
            selection_value = validity * ri
            cands.append(CandidateSentence("pf_CP_C_OPE", lbl, tpl(lbl, validity), validity, ri, selection_value))

        return PMResult(produced_cps=[cp], produced_candidates=cands, produced_scalars={"delta_ope": delta_ope})


class PM_C_LW(PM):
    def __init__(self, partition_CLW: FuzzyPartition, out_cp_key: str = "CP_C_LW"):
        super().__init__("PM_C_LW")
        self.partition = partition_CLW
        self.out_cp_key = out_cp_key

    def compute(self, data: Dict[str, Any], cp_store: Dict[str, CP]) -> PMResult:
        oee_days = data["oee_days"]
        last_week_oee_days = data["last_week_oee_days"]

        cur_week_avg = week_avg(oee_days)
        last_week_avg = week_avg(last_week_oee_days)
        delta_last = cur_week_avg - last_week_avg
        data["delta_last"] = delta_last

        cp = cp_store[self.out_cp_key]
        cp.set_degrees(self.partition.degrees(delta_last, cp.labels))

        def tpl(lbl: str, validity: float) -> str:
            return f"Compared to last week, the OEE value has {lbl}."
        

        cands: List[CandidateSentence] = []
        for lbl, wi in zip(cp.labels, cp.w):
            validity = float(wi)
            ri = float(cp.relevance_of(lbl))
            selection_value = validity * ri
            cands.append(CandidateSentence("pf_CP_C_LW", lbl, tpl(lbl, validity), validity, ri, selection_value))

        return PMResult(produced_cps=[cp], produced_candidates=cands, produced_scalars={"delta_last": delta_last})


class PM_Trend(PM):
    def __init__(self, cfg: OEEConfig, partition_TREND: FuzzyPartition, out_cp_key: str = "CP_Trend"):
        super().__init__("PM_Trend")
        self.cfg = cfg
        self.partition = partition_TREND
        self.out_cp_key = out_cp_key

    def compute(self, data: Dict[str, Any], cp_store: Dict[str, CP]) -> PMResult:

        if "trend_z" not in data or data["trend_z"] is None:
            raise ValueError(
                "trend_z not found. First, generate the SGTrend output (weekly_trends_sg.xlsx) and inject it into the main flow for the relevant week."
            )
            
        z = float(data["trend_z"])

        cp = cp_store[self.out_cp_key]
        cp.set_degrees(self.partition.degrees(z, cp.labels))

        def tpl(lbl: str, validity: float) -> str:
            return f"This week, the OEE trend shows a {lbl} pattern."
    

        cands: List[CandidateSentence] = []
        for lbl, wi in zip(cp.labels, cp.w):
            validity = float(wi)
            ri = float(cp.relevance_of(lbl))
            selection_value = validity * ri
            cands.append(CandidateSentence("pf_CP_Trend", lbl, tpl(lbl, validity), validity, ri, selection_value))

        return PMResult(produced_cps=[cp], produced_candidates=cands, produced_scalars={"trend_z": z})


class PM_Var(PM):
    def __init__(self, partition_VAR: FuzzyPartition, out_cp_key: str = "CP_Var"):
        super().__init__("PM_Var")
        self.partition = partition_VAR
        self.out_cp_key = out_cp_key

    def compute(self, data: Dict[str, Any], cp_store: Dict[str, CP]) -> PMResult:

        if "std" not in data or data["std"] is None:
            raise ValueError(
                "std not found. First, generate the SGTrend output (weekly_trends_sg.xlsx) and inject it into the main flow for the relevant week."
            )
           
        std = float(data["std"])

        cp = cp_store[self.out_cp_key]
        cp.set_degrees(self.partition.degrees(std, cp.labels))

        def tpl(lbl: str, validity: float) -> str:
            return f"The OEE trend variability is at the {lbl} level ."
          

        cands: List[CandidateSentence] = []
        for lbl, wi in zip(cp.labels, cp.w):
            validity = float(wi)
            ri = float(cp.relevance_of(lbl))
            selection_value = validity * ri
            cands.append(CandidateSentence("pf_CP_Var", lbl, tpl(lbl, validity), validity, ri, selection_value))

        return PMResult(produced_cps=[cp], produced_candidates=cands, produced_scalars={"std": std})


class PM_Process(PM):
    def __init__(self, cfg: OEEConfig, trend_cp_key: str = "CP_Trend", var_cp_key: str = "CP_Var", out_cp_key: str = "CP_Process"):
        super().__init__("PM_Process")
        self.cfg = cfg
        self.trend_cp_key = trend_cp_key
        self.var_cp_key = var_cp_key
        self.out_cp_key = out_cp_key

    def compute(self, data: Dict[str, Any], cp_store: Dict[str, CP]) -> PMResult:
        CP_Trend = cp_store[self.trend_cp_key]
        cp_var = cp_store[self.var_cp_key]
        CP_Process = cp_store[self.out_cp_key]

        out = {label: 0.0 for label in CP_Process.labels}

        for rule in self.cfg.MAMDANI_RULES:
            mu_t = CP_Trend.degree_of(rule["trend"])
            mu_v = cp_var.degree_of(rule["var"])
            activation = min(mu_t, mu_v)
            out[rule["out"]] = max(out[rule["out"]], activation)

        CP_Process.set_degrees([out[lbl] for lbl in CP_Process.labels])

        def tpl(lbl: str, validity: float) -> str:
            return f"According to these indicators, the production process is evaluated as {lbl}."



        cands: List[CandidateSentence] = []
        for lbl, wi in zip(CP_Process.labels, CP_Process.w):
            validity = float(wi)
            ri = float(CP_Process.relevance_of(lbl))
            selection_value = validity * ri
            cands.append(CandidateSentence("pf_CP_Process", lbl, tpl(lbl, validity), validity, ri, selection_value))

        process_label = CP_Process.best_label_wr(rel_threshold=0.0)
        validity_best = CP_Process.degree_of(process_label)

        data["process_best_label"] = process_label
        data["process_validity"] = validity_best

        return PMResult(
            produced_cps=[CP_Process],
            produced_candidates=cands,
            produced_scalars={"process_best_label": process_label, "process_validity": validity_best}
        )



# ============================================================
# 8) PM ActionPriority
# ============================================================

def _mamdani_centroid_1d(
    alpha_low: float,
    alpha_mid: float,
    alpha_high: float,
    mf_low: Callable[[float], float],
    mf_mid: Callable[[float], float],
    mf_high: Callable[[float], float],
    y_min: float = 0.0,
    y_max: float = 10.0,
    step: float = 0.01
) -> float:
  
    # Safety: clip if negative/NaN
    def _clip01(v: float) -> float:
        try:
            v = float(v)
        except Exception:
            return 0.0
        if v != v:  # NaN
            return 0.0
        if v < 0.0:
            return 0.0
        if v > 1.0:
            return 1.0
        return v

    aL = _clip01(alpha_low)
    aM = _clip01(alpha_mid)
    aH = _clip01(alpha_high)

    num = 0.0
    den = 0.0
    y = y_min
    # Simple Riemann sum (equal step)
    while y <= y_max + 1e-12:
        mu = max(
            min(aL, mf_low(y)),
            min(aM, mf_mid(y)),
            min(aH, mf_high(y)),
        )
        num += y * mu
        den += mu
        y += step

    if den <= 0.0:
        # If no activation, return a neutral value (middle
        return (y_min + y_max) / 2.0

    return num / den


class PM_ActionPriority(PM):

    def __init__(
        self,
        out_cp_key: str = "CP_ActionPriority",
        avail_cp_key: str = "CP_Avail",
        perf_cp_key: str = "CP_Perf",
        qual_cp_key: str = "CP_Qual",
    ):
        super().__init__("PM_ActionPriority")
        self.out_cp_key = out_cp_key
        self.avail_cp_key = avail_cp_key
        self.perf_cp_key = perf_cp_key
        self.qual_cp_key = qual_cp_key

        # Output membership functions for Mamdani
        self.mf_low = TrapezoidMF(*OEEConfig.AP_OUT_LOW_TRAP)
        self.mf_mid = TriangleMF(*OEEConfig.AP_OUT_MID_TRI)
        self.mf_high = TrapezoidMF(*OEEConfig.AP_OUT_HIGH_TRAP)

        self.protoform_id = "pf_CP_ActionPriority"

        # Universe (0.0..10.0 step 0.1)
        self._y_universe = [i / 10.0 for i in range(0, 101)]

        # Tie-break: FU > FS > FQ
        self._tie_rank = {"unplanned_downtime": 0, "speed_loss": 1, "quality_loss": 2}

        self._action_to_text = {
            "unplanned_downtime": "unplanned downtime losses",
            "speed_loss": "speed losses",
            "quality_loss": "quality losses",
        }

    def _compute_focus_crisp(self, cp: CP) -> float:
        mu_in_low = float(cp.degree_of("low"))
        mu_in_mid = float(cp.degree_of("medium"))
        mu_in_high = float(cp.degree_of("high"))

        # Rule strengths
        alpha_out_high = mu_in_low
        alpha_out_mid = mu_in_mid
        alpha_out_low = mu_in_high

        num = 0.0
        den = 0.0
        for y in self._y_universe:
            mu_low = float(self.mf_low(y))
            mu_mid = float(self.mf_mid(y))
            mu_high = float(self.mf_high(y))

            # Implication: min-clipping; Aggregation: max
            mu_agg = max(
                min(alpha_out_low, mu_low),
                min(alpha_out_mid, mu_mid),
                min(alpha_out_high, mu_high),
            )

            num += y * mu_agg
            den += mu_agg

        return 0.0 if den == 0.0 else (num / den)

    def compute(self, data: Dict[str, Any], cp_store: Dict[str, CP]) -> PMResult:
        # Read input CPs
        cp_av = cp_store[self.avail_cp_key]
        cp_pf = cp_store[self.perf_cp_key]
        cp_ql = cp_store[self.qual_cp_key]

        # Trigger strengths aligned with report logic (low/mid only)
        W_U = max(float(cp_av.degree_of("low")), float(cp_av.degree_of("medium")))
        W_S = max(float(cp_pf.degree_of("low")), float(cp_pf.degree_of("medium")))
        W_Q = max(float(cp_ql.degree_of("low")), float(cp_ql.degree_of("medium")))

        triggers = {
            "unplanned_downtime": W_U,
            "speed_loss": W_S,
            "quality_loss": W_Q,
        }

        out_cp = cp_store[self.out_cp_key]
        # out_cp validity degrees = triggers (0..1)
        out_cp.w = [W_U, W_S, W_Q]

        # If nothing triggers, no candidate sentence
        if max(triggers.values()) < 0.5:
            return PMResult(produced_cps=[out_cp], produced_candidates=[], produced_scalars={})

        # Crisp scores (ranking only)
        FU_crisp = self._compute_focus_crisp(cp_av)
        FS_crisp = self._compute_focus_crisp(cp_pf)
        FQ_crisp = self._compute_focus_crisp(cp_ql)

        crisp_scores = {
            "unplanned_downtime": FU_crisp,
            "speed_loss": FS_crisp,
            "quality_loss": FQ_crisp,
        }

        # Active actions
        active = [k for k, w in triggers.items() if w >= 0.5]

        # Sort: crisp desc, then tie-break
        active_sorted = sorted(active, key=lambda k: (-float(crisp_scores.get(k, 0.0)), self._tie_rank[k]))

        parts = [self._action_to_text[k] for k in active_sorted]

        if len(parts) == 1:
            rec_text = f"Next week, you should primarily focus on {parts[0]}."
          
        elif len(parts) == 2:
            rec_text = f"Next week, you should first focus on {parts[0]}, followed by {parts[1]}."
          
        else:
            rec_text = f"Next week, you should first focus on {parts[0]}, followed by {parts[1]}, and then on {parts[2]}."
         

        crisp_str = ", ".join(f"{k}={float(crisp_scores.get(k, 0.0)):.2f}" for k in active_sorted)
        rec_text = f"{rec_text} " 
        
        validity = float(max(triggers[k] for k in active_sorted)) if active_sorted else 0.0
        relevance = 1.0
        selection_value = validity * relevance

        cand = CandidateSentence(
            protoform_id=self.protoform_id,
            label="ActionPriority",
            text=rec_text,
            validity=validity,
            relevance=relevance,
            selection_value=selection_value,
        )

        return PMResult(produced_cps=[out_cp], produced_candidates=[cand], produced_scalars={})


class PM_Day(PM):
 

    def __init__(
        self,
        *,
        partition_OEE_Day: Any,
        out_cp_key: str = "CP_Day",
    ):
        super().__init__("PM_Day")

        if partition_OEE_Day is None or not hasattr(partition_OEE_Day, "degrees"):
            raise TypeError(
                "PM_Day: 'partition_OEE_Day' must be a valid FuzzyPartition-like object (expected degrees method)."
            )
           

        self.partition_OEE_Day = partition_OEE_Day
        self.out_cp_key = out_cp_key

    def compute(self, data: Dict[str, Any], cp_store: Dict[str, CP]) -> PMResult:
        oee_days = data["oee_days"]
        n = len(oee_days)
        if n == 0:
            raise ValueError("oee_days cannot be empty.")

        cp_days = cp_store[self.out_cp_key]
        level_labels = list(cp_days.labels)

        # Daily membership matrix (days × 3): each row [μ(low), μ(medium), μ(high)]
        daily_mat = []
        for v in oee_days:
            deg = self.partition_OEE_Day.degrees(v, level_labels)
            daily_mat.append([float(mu) for mu in deg])
        cp_days.set_degrees(daily_mat)


        ## --- Generate candidate sentences for Monday–Friday (5×3=15) ---
        day_names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]  # daily_mat[0] -> Monday

        candidates: List[CandidateSentence] = []
        best_sv = -1.0
        best_candidate: Optional[CandidateSentence] = None

        n_use = min(len(daily_mat), 5)
        for d in range(n_use):
            day = day_names[d]
            row = daily_mat[d]  

            for lvl_idx, lvl in enumerate(level_labels):
                Wi = float(row[lvl_idx])
                ri = float(cp_days.relevance_of(lvl))
                sv = ri * Wi  

                cand = CandidateSentence(
                    protoform_id="pf_CP_Day",
                    label=f"{day}|{lvl}",
                    text=f"On {day}, the OEE is at the {lvl} level.", 
                    validity=Wi,
                    relevance=ri,
                    selection_value=sv,
                )
                candidates.append(cand)

                if sv > best_sv:
                    best_sv = sv
                    best_candidate = cand

        return PMResult(
            produced_cps=[cp_days],
            produced_candidates=candidates,
            produced_scalars={},
        )




class PM_SD(PM):
    def __init__(
        self,
        CP_Day_key: str,
        partition_SD,
        CP_SD_key: str = "CP_SD",
        protoform_id: str = "pf_CP_SD",
        quantifier_relevance: Optional[List[float]] = None,
    ):
        super().__init__("PM_SD")
        self.CP_Day_key = CP_Day_key
        self.partition_SD = partition_SD
        self.CP_SD_key = CP_SD_key
        self.protoform_id = protoform_id
        self.quantifier_relevance = quantifier_relevance

    def compute(self, data: Dict[str, Any], cp_store: Dict[str, CP]) -> PMResult:
        cp_days = cp_store[self.CP_Day_key]

        daily_mat = cp_days.w
        if not isinstance(daily_mat, list) or len(daily_mat) == 0:
            raise ValueError("PM_SD: CP_Day.w daily membership matrix is empty/invalid.")

        n = len(daily_mat)

        # Fuzzy cardinality: sum column-wise
        cards = {lvl: 0.0 for lvl in cp_days.labels}
        for row in daily_mat:
            if len(row) != len(cp_days.labels):
                raise ValueError("PM_SD: CP_Day.w satır boyutu etiket sayısı ile uyuşmuyor.")
            for lvl, mu in zip(cp_days.labels, row):
                cards[lvl] += float(mu)

        #  Ratios
        p_by_level = {lvl: (cards[lvl] / n) for lvl in cp_days.labels}

        
        q_labels = ["few", "most", "all"]
        q_cps = {lvl: CP(f"_q_{lvl}", q_labels) for lvl in cp_days.labels}

        if self.quantifier_relevance is not None:
            for lvl in cp_days.labels:
                q_cps[lvl].set_relevance(list(self.quantifier_relevance))

        #  Calculate quantifier degrees for each level
        for lvl in cp_days.labels:
            p_A = p_by_level[lvl]
            q_cp = q_cps[lvl]
            q_cp.set_degrees(self.partition_SD.degrees(p_A, q_cp.labels))

        # CP_SD: A = {q}|{lvl}, W = mu, R = r_A * r_Q
        CP_SD = cp_store[self.CP_SD_key]
        sd_w = []
        sd_r = []
        cands: List[CandidateSentence] = []

        for sd_label in CP_SD.labels:
            q_lbl, lvl = sd_label.split("|", 1)

            q_cp = q_cps[lvl]
            mu = float(q_cp.degree_of(q_lbl))          # W
            r_A = float(cp_days.relevance_of(lvl))     # relevance of level
            r_Q = float(q_cp.relevance_of(q_lbl))      # relevance of quantifier
            relevance = r_A * r_Q                      # R

            sd_w.append(mu)
            sd_r.append(relevance)

            k_days = round(cards[lvl], 1)
            validity = mu
            selection_value = validity * relevance

            text = (
             
                f"During {q_lbl} days of the week, the machine’s OEE was observed at the {lvl} level."

            )

            cands.append(CandidateSentence(
                self.protoform_id,
                sd_label,
                text,
                validity,
                relevance,
                selection_value
            ))

        CP_SD.set_degrees(sd_w)
        CP_SD.set_relevance(sd_r)

        best = max(cands, key=lambda c: (c.selection_value, c.validity))
        best_q, best_lvl = best.label.split("|", 1)

        return PMResult(
            produced_cps=[CP_SD],  
            produced_candidates=cands,
            produced_scalars={"cards": cards, "p_by_level": p_by_level, "best_q": best_q, "best_lvl": best_lvl},
        )
class OEE_GLMP_LDCP:
    def __init__(self, cfg: Optional[OEEConfig] = None):
        self.cfg = cfg or OEEConfig()

        # CP'ler
        self.CP_OEE     = CP("CP_OEE",     ["low", "medium", "high"])

        self.CP_Avail   = CP("CP_Avail",   ["low", "medium", "high"])
        self.CP_Perf    = CP("CP_Perf",    ["low", "medium", "high"])
        self.CP_Qual    = CP("CP_Qual",    ["low", "medium", "high"])
        self.CP_Util    = CP("CP_Util",    ["low", "medium", "high"])
        self.CP_ActionPriority = CP("CP_ActionPriority", ["nplanned_downtimw", "speed_loss", "quality_loss"])


        self.CP_C_OPE   = CP("CP_C_OPE",   ["lower", "similar", "higher"])
        self.CP_C_PLT   = CP("CP_C_PLT",   ["lower", "similar", "higher"])

        self.CP_Day     = CP("CP_Day",     ["low", "medium", "high"])
        self.CP_SD      = CP("CP_SD",      [
            "few|low", "most|low", "all|low",
            "few|medium",  "most|medium",  "all|medium",
            "few|high","most|high","all|high",
        ])
        self.CP_C_LW        = CP("CP_C_LW",        ["deteriorated", "remained unchanged", "improved"])
        self.CP_Trend         = CP("CP_Trend",         ["decreasing", "constant", "increasing"])
        self.CP_Var           = CP("CP_Var",           ["low", "medium", "high"])
        self.CP_Process       = CP("CP_Process",       ["unstable", "partly stable", "stable"])

        # ====================================================
        # Partitions
        # ====================================================

        self.partition_OEE = FuzzyPartition({
            "low":  TrapezoidMF(*self.cfg.OEE_LOW_TRAP),
            "medium":   TriangleMF(*self.cfg.OEE_MID_TRI),
            "high": TrapezoidMF(*self.cfg.OEE_HIGH_TRAP)
        })

        self.partition_OEE_Day = FuzzyPartition({
            "low":  TrapezoidMF(*self.cfg.OEE_D_LOW_TRAP),
            "medium":   TriangleMF(*self.cfg.OEE_D_MID_TRI),
            "high": TrapezoidMF(*self.cfg.OEE_D_HIGH_TRAP)
        })

        self.partition_AVAIL = FuzzyPartition({
            "low":  TrapezoidMF(*self.cfg.AVAIL_LOW_TRAP),
            "medium":   TriangleMF(*self.cfg.AVAIL_MID_TRI),
            "high": TrapezoidMF(*self.cfg.AVAIL_HIGH_TRAP)
        })

        self.partition_PERF = FuzzyPartition({
            "low":  TrapezoidMF(*self.cfg.PERF_LOW_TRAP),
            "medium":   TriangleMF(*self.cfg.PERF_MID_TRI),
            "high": TrapezoidMF(*self.cfg.PERF_HIGH_TRAP)
        })

        self.partition_QUAL = FuzzyPartition({
            "low":  TrapezoidMF(*self.cfg.QUAL_LOW_TRAP),
            "medium":   TriangleMF(*self.cfg.QUAL_MID_TRI),
            "high": TrapezoidMF(*self.cfg.QUAL_HIGH_TRAP)
        })

        self.partition_UTIL = FuzzyPartition({
            "low":  TrapezoidMF(*self.cfg.UTIL_LOW_TRAP),
            "medium":   TriangleMF(*self.cfg.UTIL_MID_TRI),
            "high": TrapezoidMF(*self.cfg.UTIL_HIGH_TRAP)
        })


        self.partition_OPE = FuzzyPartition({
            "lower":  TrapezoidMF(*self.cfg.OPE_LOWER_TRAP),
            "similar":      TriangleMF(*self.cfg.OPE_SAME_TRI),
            "higher": TrapezoidMF(*self.cfg.OPE_HIGHER_TRAP),
        })

        self.partition_PLT = FuzzyPartition({
            "lower":  TrapezoidMF(*self.cfg.PLANT_LOWER_TRAP),
            "similar":      TriangleMF(*self.cfg.PLANT_SAME_TRI),
            "higher": TrapezoidMF(*self.cfg.PLANT_HIGHER_TRAP),
        })

        self.partition_CLW = FuzzyPartition({
            "deteriorated":   TrapezoidMF(*self.cfg.CLW_DETERIOR_TRAP),
            "remained unchanged":   TriangleMF(*self.cfg.CLW_UNCHANGE_TRI),
            "improved":    TrapezoidMF(*self.cfg.CLW_IMPROVE_TRAP),
        })

        self.partition_TREND = FuzzyPartition({
            "decreasing": TrapezoidMF(*self.cfg.TREND_DEC_TRAP),
            "constant":  TriangleMF(*self.cfg.TREND_STB_TRI),
            "increasing":  TrapezoidMF(*self.cfg.TREND_INC_TRAP),
        })

        self.partition_VAR = FuzzyPartition({
            "low":  TrapezoidMF(*self.cfg.VAR_LOW_TRAP),
            "medium":  TriangleMF(*self.cfg.VAR_MID_TRI),
            "high": TrapezoidMF(*self.cfg.VAR_HIGH_TRAP),
        })

        self.partition_SD = FuzzyPartition({
            "few": TrapezoidMF(*self.cfg.SD_FEW_TRAP),
            "most":   TriangleMF(*self.cfg.SD_MOST_TRI),
            "all":    TrapezoidMF(*self.cfg.SD_ALL_TRAP),
        })

        self.template = ReportTemplate(self.cfg)

        machine_id_getter = lambda dct: str(dct["machine_id"])
        self.pm_list: List[PM] = [
            PM_OEE(self.partition_OEE, machine_id_getter, out_cp_key="CP_OEE"),

            PM_Avail(self.partition_AVAIL, out_cp_key="CP_Avail"),
            PM_Perf(self.partition_PERF, out_cp_key="CP_Perf"),
            PM_Qual(self.partition_QUAL, out_cp_key="CP_Qual"),


            PM_ActionPriority(out_cp_key="CP_ActionPriority", avail_cp_key="CP_Avail", perf_cp_key="CP_Perf", qual_cp_key="CP_Qual"),
            PM_Util(self.partition_UTIL, out_cp_key="CP_Util"),

            PM_Day(
                partition_OEE_Day=self.partition_OEE_Day,
                out_cp_key="CP_Day"
            ),
            PM_SD(
                CP_Day_key="CP_Day",
                partition_SD=self.partition_SD,
                quantifier_relevance=list(self.cfg.R_SD),
                CP_SD_key="CP_SD",
                protoform_id="pf_CP_SD",
            ),
    
            PM_C_OPE(self.partition_OPE, out_cp_key="CP_C_OPE"),
            PM_C_PLT(self.partition_PLT, out_cp_key="CP_C_PLT"),
            PM_C_LW(self.partition_CLW, out_cp_key="CP_C_LW"),
            PM_Trend(self.cfg, self.partition_TREND, out_cp_key="CP_Trend"),
            PM_Var(self.partition_VAR, out_cp_key="CP_Var"),
            PM_Process(self.cfg, trend_cp_key="CP_Trend", var_cp_key="CP_Var", out_cp_key="CP_Process"),
        ]
        # ============================================================
        # GLMP Network Metadata (requires / produces)
        # ============================================================
        self.pm_io = {
            "PM_OEE": {"requires": ["oee_days"], "produces": ["CP_OEE"]},
            "PM_Avail": {"requires": ["avail_days"], "produces": ["CP_Avail"]},
            "PM_Perf": {"requires": ["perf_days"], "produces": ["CP_Perf"]},
            "PM_Qual": {"requires": ["qual_days"], "produces": ["CP_Qual"]},
            "PM_ActionPriority": {"requires": ["CP_Avail", "CP_Perf", "CP_Qual"], "produces": ["CP_ActionPriority"]},
            "PM_Util": {"requires": ["util_days"], "produces": ["CP_Util"]},
            "PM_Day": {"requires": ["oee_days"], "produces": ["CP_Day"]},
          
            "PM_SD": {"requires": ["CP_Day"], "produces": ["CP_SD"]},
            "PM_C_OPE": {"requires": ["oee_days", "ope_days"], "produces": ["CP_C_OPE"]},
            "PM_C_PLT": {"requires": ["oee_days", "plant_days"], "produces": ["CP_C_PLT"]},
            "PM_C_LW": {"requires": ["oee_days", "last_week_oee_days"], "produces": ["CP_C_LW"]},
            "PM_Trend": {"requires": ["trend_z"], "produces": ["CP_Trend"]},
            "PM_Var": {"requires": ["std"], "produces": ["CP_Var"]},
            "PM_Process": {"requires": ["CP_Trend", "CP_Var"], "produces": ["CP_Process"]},
        }

        self.protoform_rules: Dict[str, ProtoformRule] = {
            "pf_CP_OEE":        ProtoformRule(mode="mandatory"),
            "pf_CP_Util":       ProtoformRule(mode="conditional", R_min=0.0, W_min=0.5), 
            "pf_CP_Process":          ProtoformRule(mode="conditional", R_min=0.0, W_min=0.5),
            "pf_CP_C_PLT":    ProtoformRule(mode="conditional", R_min=0.0, W_min=0.5),
            "pf_CP_C_OPE":      ProtoformRule(mode="conditional", R_min=0.0, W_min=0.5),
            "pf_CP_SD":   ProtoformRule(mode="conditional", R_min=0.0, W_min=0.5),
            "pf_CP_Trend":       ProtoformRule(mode="conditional", R_min=0.0, W_min=0.5),
            "pf_CP_Var":         ProtoformRule(mode="conditional", R_min=0.0, W_min=0.5),
            "pf_CP_C_LW":      ProtoformRule(mode="conditional", R_min=0.0, W_min=0.5),
            "pf_CP_Avail": ProtoformRule(mode="conditional", R_min=0.0, W_min=0.5),
            "pf_CP_Perf":  ProtoformRule(mode="conditional", R_min=0.0, W_min=0.5),
            "pf_CP_Qual":  ProtoformRule(mode="conditional", R_min=0.0, W_min=0.5),
            "pf_CP_ActionPriority": ProtoformRule(mode="conditional", R_min=0.0, W_min=0.5, SV_min=0.0),
        }

        self.apply_relevance_from_config(self.cfg)

    def should_include_protoform(
        self,
        protoform_id: str,
        cand: CandidateSentence,
        ctx: Dict[str, Any]
    ) -> bool:
        rule = self.protoform_rules.get(protoform_id)
        if rule is None:
            return True
        if rule.mode == "mandatory":
            return True
        if callable(rule.include_if):
            return bool(rule.include_if(cand, ctx))

        R = float(cand.relevance)
        W = float(cand.validity)
        SV = float(cand.selection_value)
        return (R > rule.R_min) and (W >= rule.W_min) and (SV > rule.SV_min)

    def explain_inclusion(self, protoform_id: str, cand: CandidateSentence, ctx: Dict[str, Any]) -> str:
        rule = self.protoform_rules.get(protoform_id)

        if rule is None:
            return "Rule not defined → included by default."
           
        if rule.mode == "mandatory":
            return "MANDATORY protoform → always included."
           
        if callable(rule.include_if):
            try:
                res = bool(rule.include_if(cand, ctx))
            except Exception as e:
                return f"Error while running include_if: {e!r} → override could not be applied."
            return f"include_if override result → {'included' if res else 'excluded'}."

        R = float(cand.relevance)
        W = float(cand.validity)
        SV = float(cand.selection_value)

        checks = [
            (f"R({R:.4f}) > R_min({rule.R_min})", R > rule.R_min),
            (f"W({W:.4f}) > W_min({rule.W_min})", W >= rule.W_min),
            (f"SV({SV:.6f}) > SV_min({rule.SV_min})", SV > rule.SV_min),
        ]
        failed = [txt for txt, ok in checks if not ok]
        if not failed:
            return "Thresholds met → included. (" + "; ".join([txt for txt, _ in checks]) + ")"
        return "Threshold not met → excluded. Failed: " + "; ".join(failed)

    def apply_relevance_from_config(self, config) -> None:
        self.CP_OEE.set_relevance(list(self.cfg.R_OEE))

        self.CP_Avail.set_relevance(list(self.cfg.R_AVAIL))
        self.CP_Perf.set_relevance(list(self.cfg.R_PERF))
        self.CP_Qual.set_relevance(list(self.cfg.R_QUAL))
        self.CP_Util.set_relevance(list(self.cfg.R_UTIL))

        self.CP_C_OPE.set_relevance(list(self.cfg.R_C_OPE))
        self.CP_C_PLT.set_relevance(list(self.cfg.R_C_PLT))

        self.CP_Day.set_relevance(list(self.cfg.R_DAY))
        self.CP_C_LW.set_relevance(list(self.cfg.R_C_LW))
        self.CP_Trend.set_relevance(list(self.cfg.R_TREND))
        self.CP_Var.set_relevance(list(self.cfg.R_VAR))
        self.CP_Process.set_relevance(list(self.cfg.R_PROCESS))


    def reset_cps(self):
        for cp in [
            self.CP_OEE,
            self.CP_Avail, self.CP_Perf, self.CP_Qual, self.CP_Util, self.CP_ActionPriority,
            self.CP_C_OPE, self.CP_C_PLT,
              
            self.CP_Day,
            self.CP_C_LW, self.CP_Trend, self.CP_Var, self.CP_Process
        ]:
            cp.reset(reset_relevance=False)
        self.apply_relevance_from_config(self.cfg)

    def build_cp_store(self) -> Dict[str, CP]:
        return {
            "CP_OEE": self.CP_OEE,
            "CP_Avail": self.CP_Avail,
            "CP_Perf": self.CP_Perf,
            "CP_Qual": self.CP_Qual,
            "CP_Util": self.CP_Util,
            "CP_ActionPriority": self.CP_ActionPriority,
            "CP_C_OPE": self.CP_C_OPE,
            "CP_C_PLT": self.CP_C_PLT,
            "CP_Day": self.CP_Day,
            "CP_SD": self.CP_SD,            
            "CP_C_LW": self.CP_C_LW,
            "CP_Trend": self.CP_Trend,
            "CP_Var": self.CP_Var,
            "CP_Process": self.CP_Process,
        }


    def _topo_sort_pm_list(self, pm_list: List[PM]) -> List[PM]:
        from collections import defaultdict, deque

        order_index = {pm.pm_id: i for i, pm in enumerate(pm_list)}

        # produced_key -> producer pm_id (first-wins to avoid nondeterminism if ambiguous)
        produced_to_pm: Dict[str, str] = {}
        for pm in pm_list:
            io = self.pm_io.get(pm.pm_id)
            if not io:
                continue
            for k in io["produces"]:
                produced_to_pm.setdefault(k, pm.pm_id)

        adj: Dict[str, List[str]] = defaultdict(list)
        indeg: Dict[str, int] = {pm.pm_id: 0 for pm in pm_list}

        # Build edges producer -> consumer for internal dependencies
        for pm in pm_list:
            io = self.pm_io.get(pm.pm_id)
            if not io:
                continue
            for req in io["requires"]:
                prod = produced_to_pm.get(req)
                if prod and prod != pm.pm_id:
                    adj[prod].append(pm.pm_id)

        for u, vs in adj.items():
            for v in vs:
                indeg[v] += 1

        zeros = [pid for pid, d in indeg.items() if d == 0]
        zeros.sort(key=lambda pid: order_index.get(pid, 10**9))
        q = deque(zeros)

        out_ids: List[str] = []
        while q:
            u = q.popleft()
            out_ids.append(u)
            for v in adj.get(u, []):
                indeg[v] -= 1
                if indeg[v] == 0:
                    q.append(v)
            # keep stability
            q = deque(sorted(list(q), key=lambda pid: order_index.get(pid, 10**9)))

        if len(out_ids) != len(pm_list):
            raise ValueError("Topological sort failed: cycle or unresolved dependency among PMs.")

        id_to_pm = {pm.pm_id: pm for pm in pm_list}
        return [id_to_pm[pid] for pid in out_ids]

    def run_glmp_network(self, data: Dict[str, Any]) -> Tuple[Dict[str, CP], Dict[str, Any], List[CandidateSentence]]:
        cp_store = self.build_cp_store()
        all_cands: List[CandidateSentence] = []

        topo_list = self._topo_sort_pm_list(self.pm_list)

        for pm in topo_list:
            res = pm.compute(data, cp_store)
            for k, v in res.produced_scalars.items():
                data[k] = v
            all_cands.extend(res.produced_candidates)

        return cp_store, data, all_cands

    def select_best_per_protoform(self, cands: List[CandidateSentence]) -> Dict[str, CandidateSentence]:
        best: Dict[str, CandidateSentence] = {}
        for c in cands:
            if c.protoform_id not in best:
                best[c.protoform_id] = c
                continue
            b = best[c.protoform_id]
            if c.selection_value > b.selection_value:
                best[c.protoform_id] = c
            elif c.selection_value == b.selection_value and c.validity > b.validity:
                best[c.protoform_id] = c
        return best

    def generate_ldcp_report_text_and_candidates(
        self,
        oee_days: List[float],
        avail_days: List[float],
        perf_days: List[float],
        qual_days: List[float],
        util_days: List[float],
        plant_days: list[float],
        ope_days: list[float],
        last_week_oee_days: list[float],
        machine_id: str,
        trend_z: float | None = None,
        std: float | None = None,
    ) -> Tuple[str, List[CandidateSentence], List[Tuple[str, float, float, float, bool, str]]]:

        if len(oee_days) != 5:
            raise ValueError("oee_days must contain exactly 5 daily values: [Mon..Fri].")
        if len(avail_days) != 5 or len(perf_days) != 5 or len(qual_days) != 5 or len(util_days) != 5:
            raise ValueError("Avail/Perf/Qual/Util values must also contain exactly 5 daily values: [Mon..Fri].")

        self.reset_cps()

        data: Dict[str, Any] = {
            "oee_days": oee_days,
            "avail_days": avail_days,
            "perf_days": perf_days,
            "qual_days": qual_days,
            "util_days": util_days,
            "plant_days": plant_days,
            "ope_days": ope_days,
            "last_week_oee_days": last_week_oee_days,
            "machine_id": str(machine_id),
        }

        # Values from SGTrend are injected into the main flow.
        if trend_z is not None:
            data["trend_z"] = float(trend_z)
        if std is not None:
            data["std"] = float(std)

        _, _, all_cands = self.run_glmp_network(data)

        selected_best = self.select_best_per_protoform(all_cands)

        ctx = {"data": data, "machine_id": machine_id}

        audit: List[Tuple[str, float, float, float, bool, str]] = []
        for pf_id, cand in selected_best.items():
            inc = self.should_include_protoform(pf_id, cand, ctx)
            reason = self.explain_inclusion(pf_id, cand, ctx)
            audit.append((pf_id, cand.validity, cand.relevance, cand.selection_value, inc, reason))

        selected = {
            pf_id: cand
            for pf_id, cand in selected_best.items()
            if self.should_include_protoform(pf_id, cand, ctx)
        }

        report_text = self.template.format(selected, machine_id=str(machine_id))
        return report_text, all_cands, audit


# ============================================================
# 8)  Read from Excel and generate PDF + Excel(C) for each machine-week
# ============================================================

TR_MONTHS = {
    1: "JANUARY", 2: "FEBRUARY", 3: "MARCH", 4: "APRIL", 5: "MAY", 6: "JUNE",
    7: "JULY", 8: "AUGUST", 9: "SEPTEMBER", 10: "OCTOBER", 11: "NOVEMBER", 12: "DECEMBER"
}


def monday_of(d):
    return d - timedelta(days=d.weekday())

def fmt_tr(d):
    return d.strftime("%d.%m.%Y")

def month_week_index(ws):
    first = ws.replace(day=1)
    cur = monday_of(first)
    mondays = []
    for _ in range(8):
        if cur.year == ws.year and cur.month == ws.month:
            mondays.append(cur)
        cur = cur + timedelta(days=7)
    return mondays.index(ws) + 1 if ws in mondays else 1

def _unique_sheet_name(existing: set, base: str) -> str:
    base = base[:31]
    if base not in existing:
        existing.add(base)
        return base

    i = 2
    while True:
        suffix = f"_{i}"
        trimmed = base[: max(0, 31 - len(suffix))]
        name = f"{trimmed}{suffix}"
        if name not in existing:
            existing.add(name)
            return name
        i += 1


def generate_all_pdfs_from_excel(
    excel_path="upu.xlsx",
    sheet_name="RawData",
    output_dir="pdf_reports",
    pdf_title="Weekly Machine Report",
    font_path=None,
    sgtrend_excel_path="weekly_trends_sg.xlsx",
    sgtrend_sheet_name=0
):
    os.makedirs(output_dir, exist_ok=True)

    df = pd.read_excel(excel_path, sheet_name=sheet_name)

    required_cols = [
        "Date", "Machine_ID",
        "OEE", "Plant_OEE_Daily",
        "Avail", "Perf", "Qual", "OPE_OEE", "Util"
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing column(s) in Excel: {missing}")

    df["Date"] = pd.to_datetime(df["Date"]).dt.date
    df["weekday"] = pd.to_datetime(df["Date"]).dt.weekday
    df = df[df["weekday"] < 5].copy()
    df["week_start"] = df["Date"].apply(monday_of)
    df = df.sort_values(["Machine_ID", "week_start", "Date"])

    model = OEE_GLMP_LDCP(OEEConfig())

    sgtrend_map: Dict[Tuple[str, Any], Tuple[float, float]] = {}
    if sgtrend_excel_path and os.path.exists(sgtrend_excel_path):
        sg_df = pd.read_excel(sgtrend_excel_path, sheet_name=sgtrend_sheet_name)
        needed = {"Machine_ID", "WeekStart", "m_hafta", "SegmentsCount"}
        if not needed.issubset(set(sg_df.columns)):
            raise ValueError(
                f"Required columns missing in SGTrend file. Expected: {sorted(needed)} | "
                f"Received: {list(sg_df.columns)}"
            )

        sg_df["WeekStart"] = pd.to_datetime(sg_df["WeekStart"]).dt.date
        sg_df["Machine_ID"] = sg_df["Machine_ID"].astype(str)

        for _, r in sg_df.iterrows():
            key = (str(r["Machine_ID"]), r["WeekStart"])
            sgtrend_map[key] = (float(r["m_hafta"]), float(r["SegmentsCount"]))
    else:
        raise FileNotFoundError(
            f"SGTrend output file not found: {sgtrend_excel_path}. "
            "First, run the SGTrend script to generate weekly_trends_sg.xlsx."
        )

    all_excel_path = os.path.join(output_dir, "ALL_CANDIDATES.xlsx")
    with pd.ExcelWriter(all_excel_path, engine="openpyxl") as writer:
            existing_sheets: set[str] = set()

            pd.DataFrame({'info': ['initialized']}).to_excel(writer, sheet_name="INFO", index=False)
            existing_sheets.add("INFO")

            for machine_id, g_m in df.groupby("Machine_ID", sort=True):
                weeks = {ws: g.copy() for ws, g in g_m.groupby("week_start", sort=True)}

                for ws in sorted(weeks.keys()):
                    prev_ws = ws - timedelta(days=7)

                    g_week = weeks.get(ws)
                    g_prev = weeks.get(prev_ws)

                    if g_week is None or len(g_week) != 5:
                        continue
                    if g_prev is None or len(g_prev) != 5:
                        continue

                    g_week = g_week.sort_values("Date")
                    g_prev = g_prev.sort_values("Date")

                    oee_days = g_week["OEE"].astype(float).tolist()
                    avail_days = g_week["Avail"].astype(float).tolist()
                    perf_days = g_week["Perf"].astype(float).tolist()
                    qual_days = g_week["Qual"].astype(float).tolist()
                    util_days = g_week["Util"].astype(float).tolist()


                    plant_days = g_week["Plant_OEE_Daily"].astype(float).tolist()
                    ope_days = g_week["OPE_OEE"].astype(float).tolist()
                    last_week_oee_days = g_prev["OEE"].astype(float).tolist()
                  
                    sg_key = (str(machine_id), ws)
                    if sg_key not in sgtrend_map:
                        print(f"Warning: SGTrend value missing, skipped: Machine_ID={machine_id} WeekStart={ws}")
                        continue
                    trend_z, std = sgtrend_map[sg_key]

                    report_text, all_cands, audit = model.generate_ldcp_report_text_and_candidates(
                        oee_days=oee_days,
                        avail_days=avail_days,
                        perf_days=perf_days,
                        qual_days=qual_days,
                        util_days=util_days,
                        plant_days=plant_days,
                        ope_days=ope_days,
                        last_week_oee_days=last_week_oee_days,
                        trend_z=trend_z,
                        std=std,
                        machine_id=str(machine_id)
                    )

                    cands_df = candidates_to_df(all_cands)
                    audit_df = audit_to_df(audit)

                    week_end = ws + timedelta(days=4)
                    header = (
                        f"Machine ID: {machine_id}\n"
                        f"Analyzed Period: {fmt_tr(ws)} - {fmt_tr(week_end)}\n"
                    )

                    month_name = TR_MONTHS[ws.month]
                    week_no = month_week_index(ws)

                    filename = f"{machine_id}_{week_no}. HAFTA_{month_name}_{ws.year}.pdf"
                    out_path = os.path.join(output_dir, filename)

                    full_text = header + "\n\n" + report_text

                    report_text_to_pdf(
                        report_text=full_text,
                        output_path=out_path,
                        font_path=font_path,
                        title=pdf_title
                    )

                    base_sheet_c = f"{machine_id}_{ws.strftime('%Y%m%d')}_C"
                    sheet_name_final_c = _unique_sheet_name(existing_sheets, base_sheet_c)
                    cands_df.to_excel(writer, sheet_name=sheet_name_final_c, index=False)

                    base_sheet_a = f"{machine_id}_{ws.strftime('%Y%m%d')}_A"
                    sheet_name_final_a = _unique_sheet_name(existing_sheets, base_sheet_a)
                    audit_df.to_excel(writer, sheet_name=sheet_name_final_a, index=False)

                    print(f"Generated: PDF={filename} | ExcelSheet(C)={sheet_name_final_c} | ExcelSheet(A)={sheet_name_final_a}")

    print(f"\nCandidate sentences + audit written to Excel: {all_excel_path}")
    print(f"All reports generated. Output folder: {output_dir}")


if __name__ == "__main__":
    excel_path = "upu.xlsx"
    font_path = r"C:\Windows\Fonts\arial.ttf" 

    generate_all_pdfs_from_excel(
        excel_path=excel_path,
        sheet_name="RawData",
        output_dir="pdf_reports",
        pdf_title="Weekly Machine Report",
        font_path=font_path,
        sgtrend_excel_path="weekly_trends_sg.xlsx",
        sgtrend_sheet_name=0
    )

Generated: PDF=BIGLIA B 301 SM 1_4. HAFTA_JULY_2025.pdf | ExcelSheet(C)=BIGLIA B 301 SM 1_20250728_C | ExcelSheet(A)=BIGLIA B 301 SM 1_20250728_A
Generated: PDF=BIGLIA B 301 SM 1_1. HAFTA_AUGUST_2025.pdf | ExcelSheet(C)=BIGLIA B 301 SM 1_20250804_C | ExcelSheet(A)=BIGLIA B 301 SM 1_20250804_A
Generated: PDF=BIGLIA B 301 SM 1_2. HAFTA_AUGUST_2025.pdf | ExcelSheet(C)=BIGLIA B 301 SM 1_20250811_C | ExcelSheet(A)=BIGLIA B 301 SM 1_20250811_A
Generated: PDF=BIGLIA B 301 SM 1_3. HAFTA_AUGUST_2025.pdf | ExcelSheet(C)=BIGLIA B 301 SM 1_20250818_C | ExcelSheet(A)=BIGLIA B 301 SM 1_20250818_A
Generated: PDF=BIGLIA B 301 SM 1_3. HAFTA_SEPTEMBER_2025.pdf | ExcelSheet(C)=BIGLIA B 301 SM 1_20250915_C | ExcelSheet(A)=BIGLIA B 301 SM 1_20250915_A
Generated: PDF=BIGLIA B 301 SM 1_4. HAFTA_SEPTEMBER_2025.pdf | ExcelSheet(C)=BIGLIA B 301 SM 1_20250922_C | ExcelSheet(A)=BIGLIA B 301 SM 1_20250922_A
Generated: PDF=BIGLIA B 301 SM 1_5. HAFTA_SEPTEMBER_2025.pdf | ExcelSheet(C)=BIGLIA B 301 SM 1_20250929_C | 

Generated: PDF=GLEASON-PFAUTER GP130-2 BCMGE002_1. HAFTA_OCTOBER_2025.pdf | ExcelSheet(C)=GLEASON-PFAUTER GP130-2 BCMG_21 | ExcelSheet(A)=GLEASON-PFAUTER GP130-2 BCMG_22
Generated: PDF=GLEASON-PFAUTER GP130-2 BCMGE002_2. HAFTA_OCTOBER_2025.pdf | ExcelSheet(C)=GLEASON-PFAUTER GP130-2 BCMG_23 | ExcelSheet(A)=GLEASON-PFAUTER GP130-2 BCMG_24
Generated: PDF=GLEASON-PFAUTER GP130-2 BCMGE002_3. HAFTA_OCTOBER_2025.pdf | ExcelSheet(C)=GLEASON-PFAUTER GP130-2 BCMG_25 | ExcelSheet(A)=GLEASON-PFAUTER GP130-2 BCMG_26
Generated: PDF=GLEASON-PFAUTER GP130-2 BCMGE002_2. HAFTA_NOVEMBER_2025.pdf | ExcelSheet(C)=GLEASON-PFAUTER GP130-2 BCMG_27 | ExcelSheet(A)=GLEASON-PFAUTER GP130-2 BCMG_28
Generated: PDF=GLEASON-PFAUTER GP130-2 BCMGE002_3. HAFTA_NOVEMBER_2025.pdf | ExcelSheet(C)=GLEASON-PFAUTER GP130-2 BCMG_29 | ExcelSheet(A)=GLEASON-PFAUTER GP130-2 BCMG_30
Generated: PDF=GLEASON-PFAUTER GP130-2 BCMGE002_4. HAFTA_NOVEMBER_2025.pdf | ExcelSheet(C)=GLEASON-PFAUTER GP130-2 BCMG_31 | ExcelSheet(A)=GLEASON-P

Generated: PDF=HYUNDAI WIA L-210LA BCML001_2. HAFTA_NOVEMBER_2025.pdf | ExcelSheet(C)=HYUNDAI WIA L-210LA BCML001__23 | ExcelSheet(A)=HYUNDAI WIA L-210LA BCML001__24
Generated: PDF=HYUNDAI WIA L-210LA BCML001_3. HAFTA_NOVEMBER_2025.pdf | ExcelSheet(C)=HYUNDAI WIA L-210LA BCML001__25 | ExcelSheet(A)=HYUNDAI WIA L-210LA BCML001__26
Generated: PDF=HYUNDAI WIA L-210LA BCML001_4. HAFTA_NOVEMBER_2025.pdf | ExcelSheet(C)=HYUNDAI WIA L-210LA BCML001__27 | ExcelSheet(A)=HYUNDAI WIA L-210LA BCML001__28
Generated: PDF=HYUNDAI WIA L-210LA BCML001_1. HAFTA_DECEMBER_2025.pdf | ExcelSheet(C)=HYUNDAI WIA L-210LA BCML001__29 | ExcelSheet(A)=HYUNDAI WIA L-210LA BCML001__30
Generated: PDF=HYUNDAI WIA L-210LA BCML001_2. HAFTA_DECEMBER_2025.pdf | ExcelSheet(C)=HYUNDAI WIA L-210LA BCML001__31 | ExcelSheet(A)=HYUNDAI WIA L-210LA BCML001__32
Generated: PDF=HYUNDAI WIA SE 2200LM-2 BCML002_2. HAFTA_JULY_2025.pdf | ExcelSheet(C)=HYUNDAI WIA SE 2200LM-2 BCML002 | ExcelSheet(A)=HYUNDAI WIA SE 2200LM-2 BCML0_2
Gene

Generated: PDF=LIEBHERR L300 BCMGE010_4. HAFTA_NOVEMBER_2025.pdf | ExcelSheet(C)=LIEBHERR L300 BCMGE010_20251124 | ExcelSheet(A)=LIEBHERR L300 BCMGE010_202511_4
Generated: PDF=LIEBHERR L300 BCMGE010_3. HAFTA_DECEMBER_2025.pdf | ExcelSheet(C)=LIEBHERR L300 BCMGE010_20251215 | ExcelSheet(A)=LIEBHERR L300 BCMGE010_202512_2
Generated: PDF=LIEBHERR L300 BCMGE010_4. HAFTA_DECEMBER_2025.pdf | ExcelSheet(C)=LIEBHERR L300 BCMGE010_20251222 | ExcelSheet(A)=LIEBHERR L300 BCMGE010_202512_3
Generated: PDF=LIEBHERR PLC-255 BCMGE004_4. HAFTA_JULY_2025.pdf | ExcelSheet(C)=LIEBHERR PLC-255 BCMGE004_20250 | ExcelSheet(A)=LIEBHERR PLC-255 BCMGE004_202_2
Generated: PDF=LIEBHERR PLC-255 BCMGE004_1. HAFTA_SEPTEMBER_2025.pdf | ExcelSheet(C)=LIEBHERR PLC-255 BCMGE004_202_3 | ExcelSheet(A)=LIEBHERR PLC-255 BCMGE004_202_4
Generated: PDF=LIEBHERR PLC-255 BCMGE004_2. HAFTA_SEPTEMBER_2025.pdf | ExcelSheet(C)=LIEBHERR PLC-255 BCMGE004_202_5 | ExcelSheet(A)=LIEBHERR PLC-255 BCMGE004_202_6
Generated: PDF=LIEBHERR PLC

Generated: PDF=PFAUTER RA-200-2 BCMGE005_3. HAFTA_DECEMBER_2025.pdf | ExcelSheet(C)=PFAUTER RA-200-2 BCMGE005_20_12 | ExcelSheet(A)=PFAUTER RA-200-2 BCMGE005_20_13
Generated: PDF=YOU-JI YH-21 BCML003_4. HAFTA_JULY_2025.pdf | ExcelSheet(C)=YOU-JI YH-21 BCML003_20250728_C | ExcelSheet(A)=YOU-JI YH-21 BCML003_20250728_A
Generated: PDF=YOU-JI YH-21 BCML003_1. HAFTA_AUGUST_2025.pdf | ExcelSheet(C)=YOU-JI YH-21 BCML003_20250804_C | ExcelSheet(A)=YOU-JI YH-21 BCML003_20250804_A
Generated: PDF=YOU-JI YH-21 BCML003_2. HAFTA_AUGUST_2025.pdf | ExcelSheet(C)=YOU-JI YH-21 BCML003_20250811_C | ExcelSheet(A)=YOU-JI YH-21 BCML003_20250811_A
Generated: PDF=YOU-JI YH-21 BCML003_3. HAFTA_AUGUST_2025.pdf | ExcelSheet(C)=YOU-JI YH-21 BCML003_20250818_C | ExcelSheet(A)=YOU-JI YH-21 BCML003_20250818_A
Generated: PDF=YOU-JI YH-21 BCML003_4. HAFTA_AUGUST_2025.pdf | ExcelSheet(C)=YOU-JI YH-21 BCML003_20250825_C | ExcelSheet(A)=YOU-JI YH-21 BCML003_20250825_A
Generated: PDF=YOU-JI YH-21 BCML003_3. HAFTA_SEPTEMBE